In [2]:
# Setup — load all dependencies
import sys
import os

# Fix path — go up one level from notebooks/ to credit-committee/
sys.path.insert(0, os.path.abspath('..'))

import json
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv(os.path.join('..', '.env'))

# Load completed modules
from src.data.financials import get_financial_ratios
from src.data.edgar import get_filing
from src.data.transcripts import get_transcript
from src.data.resolver import resolve_ticker
from src.rag.indexer import build_filing_index, query_filing
from src.schemas.models import (
    ResearchAnalystOutput,
    ManagementTone,
    RiskFactor
)

# Initialize LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.1
)

print("All imports successful")

All imports successful


In [3]:
# Load AAPL data for testing
# We already have AAPL filing downloaded and indexed from notebook 1

ticker = "AAPL"

# Step 1: Resolve ticker
print("Resolving ticker...")
resolved = resolve_ticker(ticker)
print(f"Company: {resolved['legal_name']}")
print(f"CIK: {resolved['cik']}")
print(f"Status: {resolved['status']}")

Resolving ticker...
Company: Apple Inc.
CIK: 0000320193
Status: {'success': True}


In [4]:
# Step 2: Load financial data and filing
print("Loading financial data...")
ratio_data = get_financial_ratios(ticker)
print(f"Ratios status: {ratio_data['status']}")

print("\nLoading filing...")
filing_data = get_filing(ticker, resolved['cik'])
print(f"Filing status: {filing_data['status']}")

print("\nLoading transcript...")
transcript_data = get_transcript(ticker)
print(f"Transcript status: {transcript_data['status']}")

print("\nBuilding filing index...")
index_data = build_filing_index(ticker, filing_data)
print(f"Index status: {index_data['status']}")

Loading financial data...
Ratios status: {'success': True}

Loading filing...
Filing status: {'download': 'success', 'file_found': True, 'total_chars': 3490033, 'mda_found': True, 'risk_factors_found': True, 'success': True}

Loading transcript...
Transcript status: {'transcript_url': 'https://stockanalysis.com/stocks/aapl/transcripts/548666-q2-2026/', 'char_count': 51669, 'success': True}

Building filing index...
Loading existing index for AAPL...
Index status: {'index_source': 'loaded_existing', 'success': True}


In [5]:
# Step 3: Run the 6 filing queries and collect results
FILING_QUERIES = [
    "What is the company's primary business model and main revenue sources?",
    "What does management say about liquidity, cash position, and ability to meet near-term obligations?",
    "What debt facilities, credit agreements, and covenants does the company have?",
    "Is there any going concern language from the auditor or management?",
    "What are the top risk factors disclosed by management?",
    "Has management changed language around any key risks compared to prior periods?"
]

retriever = index_data["retriever"]
query_results = {}

for i, query in enumerate(FILING_QUERIES):
    result = query_filing(retriever, query)
    query_results[f"Q{i+1}"] = result
    print(f"\nQ{i+1}: {query}")
    print(f"Result preview: {result[:200]}")
    print("---")


Q1: What is the company's primary business model and main revenue sources?
Result preview: It may be necessary in the future to seek or renew licenses relating to various aspects of the Company s products, processes and services. While the Company has generally been able to obtain such lice
---

Q2: What does management say about liquidity, cash position, and ability to meet near-term obligations?
Result preview: | 2025 Form 10-K | 43 Term Debt The Company has outstanding Notes, which are senior unsecured obligations with interest payable in arrears. The following table provides a summary of the Company s term
---

Q3: What debt facilities, credit agreements, and covenants does the company have?
Result preview: | 2025 Form 10-K | 43 Term Debt The Company has outstanding Notes, which are senior unsecured obligations with interest payable in arrears. The following table provides a summary of the Company s term
---

Q4: Is there any going concern language from the auditor or management?


In [6]:
# Step 4: Build the Research Analyst prompt and call LLM

# Format ratio summary for context
ratios = ratio_data["ratios"]
ratio_summary = f"""
Debt/EBITDA: {ratios.get('debt_to_ebitda', 'N/A')}x
Interest Coverage: {ratios.get('interest_coverage', 'N/A')}x
Current Ratio: {ratios.get('current_ratio', 'N/A')}x
FCF to Debt: {ratios.get('fcf_to_debt', 'N/A')}x
Altman Z-Score: {ratios.get('altman_z', 'N/A')} ({ratios.get('altman_zone', 'N/A')})
"""

# Build context from query results
filing_context = "\n\n".join([
    f"{k}: {v[:500]}" for k, v in query_results.items()
])

# Transcript context
transcript_context = transcript_data.get("transcript_text", "")[:3000] if transcript_data["status"]["success"] else "Transcript unavailable"

# Schema description for LLM
schema_description = """
Output a JSON object with these exact fields:
{
  "business_summary": "200 word max summary of business model and revenue sources",
  "liquidity_assessment": "management stated liquidity position and cash runway",
  "covenant_observations": "key debt covenants thresholds and headroom if disclosed",
  "going_concern_flag": false,
  "risk_factors": [
    {"rank": 1, "description": "risk description", "source": "where in filing"},
    {"rank": 2, "description": "risk description", "source": "where in filing"},
    {"rank": 3, "description": "risk description", "source": "where in filing"},
    {"rank": 4, "description": "risk description", "source": "where in filing"},
    {"rank": 5, "description": "risk description", "source": "where in filing"}
  ],
  "management_tone": "Confident",
  "language_changes": "notable changes vs prior filing or null",
  "transcript_available": true
}

management_tone must be one of: Confident, Cautious, Defensive, Distressed
going_concern_flag must be true or false
Output ONLY the JSON object. No preamble, no markdown backticks.
"""

# Build full prompt
system_prompt = """You are a senior credit research analyst at a major bank.
Your job is to extract specific qualitative credit risk signals from SEC filings 
and earnings call transcripts. Be precise, factual, and reference specific 
sections. Output ONLY valid JSON — no preamble, no markdown backticks."""

user_prompt = f"""
Company: {resolved['legal_name']}
Sector: {resolved['sector']}
Ticker: {ticker}

FINANCIAL RATIO SUMMARY:
{ratio_summary}

FILING EXCERPTS (from SEC 10-K):
{filing_context}

EARNINGS CALL TRANSCRIPT (most recent):
{transcript_context}

{schema_description}
"""

print("Calling LLM...")
print(f"Prompt length: {len(user_prompt):,} chars")

from langchain_core.messages import SystemMessage, HumanMessage

response = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content=user_prompt)
])

print("\nLLM Response:")
print(response.content)

Calling LLM...
Prompt length: 7,382 chars

LLM Response:
{
  "business_summary": "Apple Inc. is a technology company with revenue sources from the sale of products such as iPhones, Macs, and iPads, as well as services like Apple Music and Apple TV+. The company has historically experienced higher net sales in its first quarter compared to other quarters due to business seasonality and product introductions.",
  "liquidity_assessment": "Not explicitly stated in the provided excerpts, but the company's strong financial performance and cash position suggest a stable liquidity position.",
  "covenant_observations": "The company has outstanding senior unsecured notes with interest payable in arrears, but specific covenant thresholds and headroom are not disclosed in the provided excerpts.",
  "going_concern_flag": false,
  "risk_factors": [
    {"rank": 1, "description": "Regulatory changes and other actions that materially adversely affect the Company's business", "source": "Q6: Regulatory

In [7]:
# Step 5: Parse and validate with Pydantic + retry logic

def parse_with_retry(llm_response: str, max_retries: int = 2) -> ResearchAnalystOutput:
    """
    Attempts to parse LLM response as ResearchAnalystOutput.
    Retries up to max_retries times if validation fails.
    """
    
    last_error = None
    
    for attempt in range(max_retries + 1):
        try:
            # Clean response — remove markdown backticks if present
            clean = llm_response.strip()
            if clean.startswith("```"):
                clean = clean.split("```")[1]
                if clean.startswith("json"):
                    clean = clean[4:]
            clean = clean.strip()
            
            # Fix common LLM issue — "null" string instead of None
            clean = clean.replace('"null"', 'null')
            
            # Parse JSON
            data = json.loads(clean)
            
            # Validate with Pydantic
            output = ResearchAnalystOutput(**data)
            print(f"Validation PASSED on attempt {attempt + 1}")
            return output
            
        except json.JSONDecodeError as e:
            last_error = f"JSON parse error: {e}"
            print(f"Attempt {attempt + 1} failed — {last_error}")
            
        except Exception as e:
            last_error = f"Pydantic validation error: {e}"
            print(f"Attempt {attempt + 1} failed — {last_error}")
            
            if attempt < max_retries:
                # Retry with error feedback
                print(f"Retrying with error feedback...")
                retry_prompt = f"""Your previous output failed validation with this error:
{last_error}

Please fix your output and return ONLY valid JSON matching the schema.
Previous output was:
{llm_response}

{schema_description}"""
                
                response = llm.invoke([
                    SystemMessage(content=system_prompt),
                    HumanMessage(content=retry_prompt)
                ])
                llm_response = response.content
                print(f"New LLM response: {llm_response[:200]}")
    
    raise RuntimeError(
        f"Research Analyst failed after {max_retries + 1} attempts. "
        f"Last error: {last_error}"
    )


# Test parsing
output = parse_with_retry(response.content)

print("\nParsed Output:")
print(f"Business Summary: {output.business_summary[:100]}...")
print(f"Going Concern: {output.going_concern_flag}")
print(f"Management Tone: {output.management_tone}")
print(f"Transcript Available: {output.transcript_available}")
print(f"Risk Factors:")
for rf in output.risk_factors:
    print(f"  {rf.rank}. {rf.description[:80]}")

Validation PASSED on attempt 1

Parsed Output:
Business Summary: Apple Inc. is a technology company with revenue sources from the sale of products such as iPhones, M...
Going Concern: False
Management Tone: ManagementTone.CONFIDENT
Transcript Available: True
Risk Factors:
  1. Regulatory changes and other actions that materially adversely affect the Compan
  2. Uncertain tax positions
  3. Difficulty in obtaining licenses on commercially reasonable terms
  4. Macroeconomic conditions, tariffs, and other measures
  5. Legal and regulatory proceedings


In [8]:
# Step 6: Complete run_research_analyst function

def run_research_analyst(state: dict) -> dict:
    """
    Runs the Research Analyst agent.
    Input: state (dict) — LangGraph pipeline state
    Output: updated state dict with research_analyst_output populated
    """
    
    ticker      = state["ticker"]
    legal_name  = state["legal_name"]
    sector      = state["sector"]
    
    # --- Build filing index ---
    filing_data = {
        "mda":          state.get("mda"),
        "risk_factors": state.get("risk_factors"),
        "full_text":    state.get("full_text")
    }
    
    index_data = build_filing_index(ticker, filing_data)
    
    if not index_data["status"]["success"]:
        state["errors"].append(
            f"Research Analyst: index build failed — "
            f"{index_data['status'].get('error')}"
        )
        state["current_stage"] = "research_analyst_failed"
        return state
    
    retriever = index_data["retriever"]
    
    # --- Run filing queries ---
    FILING_QUERIES = [
        "What is the company's primary business model and main revenue sources?",
        "What does management say about liquidity, cash position, and ability to meet near-term obligations?",
        "What debt facilities, credit agreements, and covenants does the company have?",
        "Is there any going concern language from the auditor or management?",
        "What are the top risk factors disclosed by management?",
        "Has management changed language around any key risks compared to prior periods?"
    ]
    
    query_results = {}
    for i, query in enumerate(FILING_QUERIES):
        try:
            query_results[f"Q{i+1}"] = query_filing(retriever, query)
        except Exception as e:
            query_results[f"Q{i+1}"] = f"Query failed: {str(e)}"
    
    # --- Build context ---
    ratios = state.get("ratio_table", {}).get("ratios", {})
    ratio_summary = f"""
Debt/EBITDA: {ratios.get('debt_to_ebitda', 'N/A')}x
Interest Coverage: {ratios.get('interest_coverage', 'N/A')}x
Current Ratio: {ratios.get('current_ratio', 'N/A')}x
FCF to Debt: {ratios.get('fcf_to_debt', 'N/A')}x
Altman Z-Score: {ratios.get('altman_z', 'N/A')} ({ratios.get('altman_zone', 'N/A')})
"""
    
    filing_context = "\n\n".join([
        f"{k}: {v[:500]}" for k, v in query_results.items()
    ])
    
    transcript_text = state.get("transcript_text", "")
    transcript_available = bool(transcript_text)
    transcript_context = transcript_text[:3000] if transcript_available else "Transcript unavailable"
    
    # --- Schema description ---
    schema_description = """
Output a JSON object with these exact fields:
{
  "business_summary": "200 word max summary of business model and revenue sources",
  "liquidity_assessment": "management stated liquidity position and cash runway",
  "covenant_observations": "key debt covenants thresholds and headroom if disclosed",
  "going_concern_flag": false,
  "risk_factors": [
    {"rank": 1, "description": "risk description", "source": "where in filing"},
    {"rank": 2, "description": "risk description", "source": "where in filing"},
    {"rank": 3, "description": "risk description", "source": "where in filing"},
    {"rank": 4, "description": "risk description", "source": "where in filing"},
    {"rank": 5, "description": "risk description", "source": "where in filing"}
  ],
  "management_tone": "Confident",
  "language_changes": null,
  "transcript_available": true
}
management_tone must be one of: Confident, Cautious, Defensive, Distressed
going_concern_flag must be true or false
Output ONLY the JSON object. No preamble, no markdown backticks."""

    system_prompt = """You are a senior credit research analyst at a major bank.
Your job is to extract specific qualitative credit risk signals from SEC filings
and earnings call transcripts. Be precise, factual, and reference specific
sections. Output ONLY valid JSON — no preamble, no markdown backticks."""

    user_prompt = f"""
Company: {legal_name}
Sector: {sector}
Ticker: {ticker}

FINANCIAL RATIO SUMMARY:
{ratio_summary}

FILING EXCERPTS (from SEC 10-K):
{filing_context}

EARNINGS CALL TRANSCRIPT (most recent):
{transcript_context}

{schema_description}
"""

    # --- Call LLM with retry ---
    last_error = None
    llm_response = None
    output = None

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ])
    llm_response = response.content

    for attempt in range(3):
        try:
            clean = llm_response.strip()
            if clean.startswith("```"):
                clean = clean.split("```")[1]
                if clean.startswith("json"):
                    clean = clean[4:]
            clean = clean.strip()
            clean = clean.replace('"null"', 'null')

            data   = json.loads(clean)
            output = ResearchAnalystOutput(**data)
            break

        except Exception as e:
            last_error = str(e)
            if attempt < 2:
                retry_response = llm.invoke([
                    SystemMessage(content=system_prompt),
                    HumanMessage(content=f"""Your output failed validation: {last_error}
Fix and return ONLY valid JSON.
Previous output: {llm_response}
{schema_description}""")
                ])
                llm_response = retry_response.content

    if output is None:
        state["errors"].append(
            f"Research Analyst failed after 3 attempts: {last_error}"
        )
        state["current_stage"] = "research_analyst_failed"
        return state

    # --- Write to state ---
    state["research_analyst_output"] = output.model_dump()
    state["current_stage"] = "research_analyst_complete"
    
    print(f"Research Analyst complete — tone: {output.management_tone}, "
          f"going concern: {output.going_concern_flag}")
    
    return state


# --- Test with simulated state ---
test_state = {
    "ticker":         "AAPL",
    "legal_name":     resolved["legal_name"],
    "sector":         resolved["sector"],
    "mda":            filing_data["mda"],
    "risk_factors":   filing_data["risk_factors"],
    "full_text":      filing_data["full_text"],
    "transcript_text": transcript_data["transcript_text"],
    "ratio_table":    ratio_data,
    "errors":         [],
    "warnings":       []
}

updated_state = run_research_analyst(test_state)

print(f"\nStage: {updated_state['current_stage']}")
print(f"Errors: {updated_state['errors']}")
print(f"\nOutput keys: {list(updated_state['research_analyst_output'].keys())}")
print(f"\nBusiness Summary: {updated_state['research_analyst_output']['business_summary'][:200]}")

Loading existing index for AAPL...
Research Analyst complete — tone: ManagementTone.CONFIDENT, going concern: False

Stage: research_analyst_complete
Errors: []

Output keys: ['business_summary', 'liquidity_assessment', 'covenant_observations', 'going_concern_flag', 'risk_factors', 'management_tone', 'language_changes', 'transcript_available']

Business Summary: Apple Inc. is a technology company with revenue sources from the sale of products such as iPhones, Macs, and iPads, as well as services like Apple Music and Apple TV+. The company has historically exp


In [9]:
# Credit Analyst Agent
# Inputs: ratio_table from financials.py + research_analyst_output
# No ChromaDB queries needed — pure quantitative interpretation

from src.schemas.models import CreditAnalystOutput, InternalRating

CREDIT_ANALYST_SYSTEM_PROMPT = """You are a senior credit analyst at a major bank.
Your job is to interpret financial ratios and assign a preliminary internal credit rating.
Use the rating mapping guidelines provided.
Output ONLY valid JSON — no preamble, no markdown backticks."""

RATING_GUIDELINES = """
RATING MAPPING GUIDELINES:
AAA/AA : Debt/EBITDA < 1x, Interest Coverage > 15x, Altman Z > 5
A      : Debt/EBITDA 1-2x, Interest Coverage 8-15x, Altman Z > 4
BBB    : Debt/EBITDA 2-3x, Interest Coverage 4-8x, Altman Z > 3
BB     : Debt/EBITDA 3-5x, Interest Coverage 2-4x, Altman Z 2-3
B      : Debt/EBITDA 5-7x, Interest Coverage 1.5-2x, Altman Z 1.5-2
CCC    : Debt/EBITDA > 7x, Interest Coverage < 1.5x, Altman Z < 1.8
CC/C/D : Interest Coverage < 1x, negative FCF, imminent default risk

TREND ASSESSMENT:
- Compare most recent ratio vs prior periods
- Improving: ratio moving in positive direction over last 3 quarters
- Deteriorating: ratio moving in negative direction over last 3 quarters
- Stable: minimal change quarter over quarter
"""

CREDIT_ANALYST_SCHEMA = """
Output a JSON object with these exact fields:
{
  "debt_to_ebitda": 0.68,
  "interest_coverage": 33.83,
  "current_ratio": 0.89,
  "fcf_to_debt": 1.0,
  "net_debt_to_ebitda": 0.43,
  "dscr": 1.13,
  "altman_z": 29.77,
  "altman_zone": "SAFE",
  "leverage_trend": "Stable",
  "coverage_trend": "Stable",
  "cashflow_trend": "Stable",
  "quantitative_narrative": "300 word max interpretation of the quantitative picture",
  "primary_concern": "single metric most responsible for the rating",
  "preliminary_rating": "AAA",
  "preliminary_rating_rationale": "one paragraph explaining the rating decision"
}

preliminary_rating must be one of: AAA, AA, A, BBB, BB, B, CCC, CC, C, D
leverage_trend, coverage_trend, cashflow_trend must be: Improving, Stable, or Deteriorating
Output ONLY the JSON object. No preamble, no markdown backticks."""


def run_credit_analyst(state: dict) -> dict:
    """
    Runs the Credit Analyst agent.
    Input: state (dict) — LangGraph pipeline state
    Output: updated state dict with credit_analyst_output populated
    """

    ratio_table           = state.get("ratio_table", {})
    ratios                = ratio_table.get("ratios", {})
    research_output       = state.get("research_analyst_output", {})

    # --- Format ratio table ---
    ratio_context = f"""
FINANCIAL RATIOS (Latest Year: {ratio_table.get('latest_year', 'N/A')}):
| Metric              | Value                                    |
|---------------------|------------------------------------------|
| Debt/EBITDA         | {ratios.get('debt_to_ebitda', 'N/A')}x  |
| Interest Coverage   | {ratios.get('interest_coverage', 'N/A')}x|
| Current Ratio       | {ratios.get('current_ratio', 'N/A')}x   |
| FCF to Debt         | {ratios.get('fcf_to_debt', 'N/A')}x     |
| Net Debt/EBITDA     | {ratios.get('net_debt_to_ebitda', 'N/A')}x|
| DSCR                | {ratios.get('dscr', 'N/A')}x            |
| Altman Z-Score      | {ratios.get('altman_z', 'N/A')} ({ratios.get('altman_zone', 'N/A')})|
"""

    # --- Research Analyst summary for context ---
    research_context = f"""
RESEARCH ANALYST SUMMARY:
Business: {research_output.get('business_summary', 'N/A')[:300]}
Liquidity: {research_output.get('liquidity_assessment', 'N/A')[:200]}
Management Tone: {research_output.get('management_tone', 'N/A')}
Going Concern Flag: {research_output.get('going_concern_flag', False)}
"""

    user_prompt = f"""
Company: {state['legal_name']}
Sector: {state['sector']}
Ticker: {state['ticker']}

{ratio_context}

{research_context}

{RATING_GUIDELINES}

{CREDIT_ANALYST_SCHEMA}
"""

    # --- Call LLM with retry ---
    last_error   = None
    llm_response = llm.invoke([
        SystemMessage(content=CREDIT_ANALYST_SYSTEM_PROMPT),
        HumanMessage(content=user_prompt)
    ]).content
    output = None

    for attempt in range(3):
        try:
            clean = llm_response.strip()
            if clean.startswith("```"):
                clean = clean.split("```")[1]
                if clean.startswith("json"):
                    clean = clean[4:]
            clean  = clean.strip()
            clean  = clean.replace('"null"', 'null')
            data   = json.loads(clean)
            output = CreditAnalystOutput(**data)
            break

        except Exception as e:
            last_error = str(e)
            if attempt < 2:
                llm_response = llm.invoke([
                    SystemMessage(content=CREDIT_ANALYST_SYSTEM_PROMPT),
                    HumanMessage(content=(
                        f"Your output failed validation: {last_error}\n"
                        f"Fix and return ONLY valid JSON.\n"
                        f"Previous output: {llm_response}\n"
                        f"{CREDIT_ANALYST_SCHEMA}"
                    ))
                ]).content

    if output is None:
        state["errors"].append(
            f"Credit Analyst failed after 3 attempts: {last_error}"
        )
        state["current_stage"] = "credit_analyst_failed"
        return state

    state["credit_analyst_output"] = output.model_dump()
    state["current_stage"]         = "credit_analyst_complete"

    print(f"Credit Analyst complete — "
          f"rating: {output.preliminary_rating}, "
          f"primary concern: {output.primary_concern}")

    return state


# --- Test ---
updated_state = run_credit_analyst(updated_state)

print(f"\nStage: {updated_state['current_stage']}")
print(f"Errors: {updated_state['errors']}")
print(f"Preliminary Rating: {updated_state['credit_analyst_output']['preliminary_rating']}")
print(f"Narrative: {updated_state['credit_analyst_output']['quantitative_narrative'][:300]}")

Credit Analyst complete — rating: InternalRating.AAA, primary concern: Current Ratio

Stage: credit_analyst_complete
Errors: []
Preliminary Rating: InternalRating.AAA
Narrative: Apple Inc.'s financial ratios indicate a strong and stable financial position. The debt-to-EBITDA ratio of 0.68x and net debt-to-EBITDA ratio of 0.43x demonstrate low leverage. The interest coverage ratio of 33.83x and DSCR of 1.13x show the company's ability to meet its financial obligations. The A


In [10]:
# Devil's Advocate Agent
# Inputs: research_analyst_output + credit_analyst_output + ChromaDB
# Must disagree with Credit Analyst rating — counter_rating one notch below

from src.schemas.models import DevilsAdvocateOutput

RATING_ORDER = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC", "CC", "C", "D"]

def enforce_counter_rating(counter_rating: str, preliminary_rating: str) -> str:
    """Ensures counter_rating is at least one notch below preliminary_rating."""
    try:
        prelim_idx  = RATING_ORDER.index(preliminary_rating)
        counter_idx = RATING_ORDER.index(counter_rating)
        # Higher index = lower rating
        if counter_idx <= prelim_idx:
            # Force one notch below
            new_idx = min(prelim_idx + 1, len(RATING_ORDER) - 1)
            return RATING_ORDER[new_idx]
        return counter_rating
    except ValueError:
        return RATING_ORDER[min(RATING_ORDER.index(preliminary_rating) + 1, 9)]


DEVIL_SYSTEM_PROMPT = """You are the Devil's Advocate in a credit committee at a major bank.
Your ONLY job is to find reasons this borrower will default.
You MUST disagree with the Credit Analyst's assessment.
Ground every argument in specific filing evidence.
Produce minimum 3, maximum 6 adversarial risk arguments.
Output ONLY valid JSON — no preamble, no markdown backticks."""

ADVERSARIAL_QUERIES = [
    "What near-term debt maturities does the company face and how does it plan to refinance?",
    "What contingent liabilities, legal proceedings, or off-balance-sheet obligations exist?",
    "What assumptions underlie management's forward guidance and how sensitive are they?",
    "What macro or industry headwinds could materially impair revenue or margins?"
]

DEVIL_SCHEMA = """
Output a JSON object with these exact fields:
{
  "adversarial_risks": [
    {
      "rank": 1,
      "risk_description": "description of risk",
      "filing_evidence": "specific text from filing supporting this",
      "quantitative_impact": "estimated financial impact if risk materializes",
      "probability": "High"
    },
    {
      "rank": 2,
      "risk_description": "description of risk",
      "filing_evidence": "specific text from filing",
      "quantitative_impact": "estimated impact",
      "probability": "Medium"
    },
    {
      "rank": 3,
      "risk_description": "description of risk",
      "filing_evidence": "specific text from filing",
      "quantitative_impact": "estimated impact",
      "probability": "Low"
    }
  ],
  "most_likely_default_trigger": "single most plausible path to default in 12-24 months",
  "bear_case_narrative": "200 word max bear case summary",
  "counter_rating": "AA",
  "macro_vulnerability": "how a macro downturn specifically hurts this borrower"
}
adversarial_risks must have minimum 3 items
probability must be one of: High, Medium, Low
counter_rating must be one of: AAA, AA, A, BBB, BB, B, CCC, CC, C, D
Output ONLY the JSON object. No preamble, no markdown backticks."""


def run_devil_advocate(state: dict) -> dict:
    """
    Runs the Devil's Advocate agent.
    Input: state (dict) — LangGraph pipeline state
    Output: updated state dict with devils_advocate_output populated
    """

    ticker          = state["ticker"]
    research_output = state.get("research_analyst_output", {})
    credit_output   = state.get("credit_analyst_output", {})
    preliminary_rating = credit_output.get(
        "preliminary_rating", "BBB"
    ).replace("InternalRating.", "")

    # --- Load index and run adversarial queries ---
    filing_data = {
        "mda":          state.get("mda"),
        "risk_factors": state.get("risk_factors"),
        "full_text":    state.get("full_text")
    }
    index_data = build_filing_index(ticker, filing_data)
    retriever  = index_data["retriever"]

    adversarial_context = {}
    for i, query in enumerate(ADVERSARIAL_QUERIES):
        try:
            adversarial_context[f"AQ{i+1}"] = query_filing(retriever, query)
        except Exception as e:
            adversarial_context[f"AQ{i+1}"] = f"Query failed: {str(e)}"

    filing_evidence = "\n\n".join([
        f"{k}: {v[:400]}" for k, v in adversarial_context.items()
    ])

    user_prompt = f"""
Company: {state['legal_name']}
Sector: {state['sector']}
Ticker: {ticker}

CREDIT ANALYST PRELIMINARY RATING: {preliminary_rating}
You MUST assign a counter_rating that is LOWER (worse) than {preliminary_rating}.

RESEARCH ANALYST OUTPUT:
Business: {research_output.get('business_summary', 'N/A')[:200]}
Risk Factors: {str(research_output.get('risk_factors', []))[:300]}
Management Tone: {research_output.get('management_tone', 'N/A')}
Going Concern: {research_output.get('going_concern_flag', False)}

CREDIT ANALYST OUTPUT:
Rating: {preliminary_rating}
Primary Concern: {credit_output.get('primary_concern', 'N/A')}
Narrative: {credit_output.get('quantitative_narrative', 'N/A')[:200]}

ADVERSARIAL FILING EVIDENCE:
{filing_evidence}

MACRO CONTEXT:
{state.get('macro_narrative', 'Not available')}

{DEVIL_SCHEMA}
"""

    last_error   = None
    llm_response = llm.invoke([
        SystemMessage(content=DEVIL_SYSTEM_PROMPT),
        HumanMessage(content=user_prompt)
    ]).content
    output = None

    for attempt in range(3):
        try:
            clean = llm_response.strip()
            if clean.startswith("```"):
                clean = clean.split("```")[1]
                if clean.startswith("json"):
                    clean = clean[4:]
            clean  = clean.strip()
            clean  = clean.replace('"null"', 'null')
            data   = json.loads(clean)
            output = DevilsAdvocateOutput(**data)

            # Enforce counter_rating constraint
            output.counter_rating = enforce_counter_rating(
                str(output.counter_rating).replace("InternalRating.", ""),
                preliminary_rating
            )
            break

        except Exception as e:
            last_error = str(e)
            if attempt < 2:
                llm_response = llm.invoke([
                    SystemMessage(content=DEVIL_SYSTEM_PROMPT),
                    HumanMessage(content=(
                        f"Your output failed validation: {last_error}\n"
                        f"Fix and return ONLY valid JSON.\n"
                        f"Previous output: {llm_response}\n"
                        f"{DEVIL_SCHEMA}"
                    ))
                ]).content

    if output is None:
        state["errors"].append(
            f"Devil's Advocate failed after 3 attempts: {last_error}"
        )
        state["current_stage"] = "devils_advocate_failed"
        return state

    state["devils_advocate_output"] = output.model_dump()
    state["current_stage"]          = "devils_advocate_complete"

    print(f"Devil's Advocate complete — "
          f"counter_rating: {output.counter_rating}, "
          f"default trigger: {output.most_likely_default_trigger[:80]}")

    return state


# --- Test ---
updated_state = run_devil_advocate(updated_state)

print(f"\nStage: {updated_state['current_stage']}")
print(f"Errors: {updated_state['errors']}")
print(f"Counter Rating: {updated_state['devils_advocate_output']['counter_rating']}")
print(f"Bear Case: {updated_state['devils_advocate_output']['bear_case_narrative'][:300]}")
print(f"\nAdversarial Risks:")
for risk in updated_state['devils_advocate_output']['adversarial_risks']:
    print(f"  {risk['rank']}. {risk['risk_description'][:80]}")

Loading existing index for AAPL...
Devil's Advocate complete — counter_rating: BB, default trigger: Regulatory changes leading to a decline in iPhone sales

Stage: devils_advocate_complete
Errors: []
Counter Rating: BB
Bear Case: Apple's business is heavily reliant on iPhone sales, which are vulnerable to regulatory changes and global economic downturns. A decline in iPhone sales, combined with increasing competition in the services sector, could lead to a significant decrease in revenue and profitability, ultimately resulti

Adversarial Risks:
  1. Regulatory changes affecting business
  2. Default on senior unsecured obligations
  3. Decline in products gross margin
  4. Intense competition in the market
  5. Economic downturn impacting consumer spending
  6. Supply chain disruptions


c:\Users\Dev Vatsayan\credit-committee\venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='counter_rating', input_value='BB', input_type=str])
  return self.__pydantic_serializer__.to_python(


In [11]:
# Fix the enum serialization warning
# Replace the enforce_counter_rating call in run_devil_advocate with this:

from src.schemas.models import InternalRating

def enforce_counter_rating(counter_rating: str, preliminary_rating: str) -> InternalRating:
    """Ensures counter_rating is at least one notch below preliminary_rating."""
    try:
        prelim_idx  = RATING_ORDER.index(preliminary_rating)
        counter_idx = RATING_ORDER.index(counter_rating)
        if counter_idx <= prelim_idx:
            new_idx = min(prelim_idx + 1, len(RATING_ORDER) - 1)
            return InternalRating(RATING_ORDER[new_idx])
        return InternalRating(counter_rating)
    except ValueError:
        return InternalRating(RATING_ORDER[min(
            RATING_ORDER.index(preliminary_rating) + 1, 9
        )])

# Quick test
print(enforce_counter_rating("AAA", "AAA"))  # should return AA
print(enforce_counter_rating("B", "AAA"))    # should return B (already lower)
print(enforce_counter_rating("AA", "AAA"))   # should return AA (one notch below)

InternalRating.AA
InternalRating.B
InternalRating.AA


In [12]:
# Orchestrator Agent
# Final synthesis — combines all three agents + macro + human input

from src.schemas.models import OrchestratorOutput, get_pd_from_rating

ORCHESTRATOR_SYSTEM_PROMPT = """You are the chair of a credit committee at a major bank.
You have just heard from three analysts and must render a final credit decision.
Synthesize all inputs into a structured credit memo.
If human input is provided, weight it most heavily.
Output ONLY valid JSON — no preamble, no markdown backticks."""

APPROVE_RATINGS     = ["AAA", "AA", "A", "BBB"]
CONDITIONAL_RATINGS = ["BB", "B"]
DECLINE_RATINGS     = ["CCC", "CC", "C", "D"]

ORCHESTRATOR_SCHEMA = """
Output a JSON object with these exact fields:
{
  "company_name": "Apple Inc.",
  "ticker": "AAPL",
  "sector": "Technology",
  "analysis_date": "2026-07-04",
  "final_rating": "AAA",
  "pd_12month": null,
  "pd_3year": null,
  "top_risk_factors": [
    {"rank": 1, "factor": "factor name", "description": "detailed description", "severity": "High"},
    {"rank": 2, "factor": "factor name", "description": "detailed description", "severity": "Medium"},
    {"rank": 3, "factor": "factor name", "description": "detailed description", "severity": "Medium"},
    {"rank": 4, "factor": "factor name", "description": "detailed description", "severity": "Low"},
    {"rank": 5, "factor": "factor name", "description": "detailed description", "severity": "Low"}
  ],
  "mitigants": ["mitigant 1", "mitigant 2", "mitigant 3"],
  "lending_decision": "Approve",
  "decision_conditions": null,
  "narrative_rationale": "400 word committee chair narrative explaining the decision",
  "confidence_level": "High",
  "confidence_rationale": "one sentence explaining confidence level",
  "credit_analyst_rating": "AAA",
  "devils_advocate_rating": "AA",
  "human_input_received": false
}
final_rating must be one of: AAA, AA, A, BBB, BB, B, CCC, CC, C, D
lending_decision must be one of: Approve, Conditional, Decline
severity must be one of: High, Medium, Low
confidence_level must be one of: High, Medium, Low
pd_12month and pd_3year must be null — they will be populated after validation
Output ONLY the JSON object. No preamble, no markdown backticks."""


def determine_lending_decision(rating: str) -> str:
    """Maps final rating to lending decision."""
    rating = rating.replace("InternalRating.", "")
    if rating in APPROVE_RATINGS:
        return "Approve"
    elif rating in CONDITIONAL_RATINGS:
        return "Conditional"
    return "Decline"


def run_orchestrator(state: dict) -> dict:
    """
    Runs the Orchestrator agent.
    Input: state (dict) — LangGraph pipeline state
    Output: updated state dict with orchestrator_output populated
    """

    from datetime import date

    research_output  = state.get("research_analyst_output", {})
    credit_output    = state.get("credit_analyst_output", {})
    devil_output     = state.get("devils_advocate_output", {})
    macro_narrative  = state.get("macro_narrative", "Not available")
    human_input      = state.get("human_input", None)

    credit_rating = str(credit_output.get(
        "preliminary_rating", "BBB"
    )).replace("InternalRating.", "")

    devil_rating = str(devil_output.get(
        "counter_rating", "BB"
    )).replace("InternalRating.", "")

    human_section = (
        f"\nSENIOR ANALYST OVERRIDE (weight most heavily):\n{human_input}"
        if human_input else "\nNo human input provided."
    )

    user_prompt = f"""
Company: {state['legal_name']}
Sector: {state['sector']}
Ticker: {state['ticker']}
Analysis Date: {date.today().isoformat()}

RESEARCH ANALYST OUTPUT:
Business Summary: {research_output.get('business_summary', 'N/A')[:300]}
Liquidity: {research_output.get('liquidity_assessment', 'N/A')[:200]}
Covenants: {research_output.get('covenant_observations', 'N/A')[:200]}
Management Tone: {research_output.get('management_tone', 'N/A')}
Going Concern: {research_output.get('going_concern_flag', False)}
Top Risks: {str(research_output.get('risk_factors', []))[:400]}

CREDIT ANALYST OUTPUT:
Preliminary Rating: {credit_rating}
Primary Concern: {credit_output.get('primary_concern', 'N/A')}
Narrative: {credit_output.get('quantitative_narrative', 'N/A')[:300]}
Rationale: {credit_output.get('preliminary_rating_rationale', 'N/A')[:200]}

DEVIL'S ADVOCATE OUTPUT:
Counter Rating: {devil_rating}
Default Trigger: {devil_output.get('most_likely_default_trigger', 'N/A')}
Bear Case: {devil_output.get('bear_case_narrative', 'N/A')[:300]}
Adversarial Risks: {str(devil_output.get('adversarial_risks', []))[:400]}
Macro Vulnerability: {devil_output.get('macro_vulnerability', 'N/A')[:200]}

MACRO CONTEXT:
{macro_narrative}
{human_section}

LENDING DECISION MAPPING:
Approve     : final_rating in AAA, AA, A, BBB
Conditional : final_rating in BB, B
Decline     : final_rating in CCC, CC, C, D

{ORCHESTRATOR_SCHEMA}
"""

    last_error   = None
    llm_response = llm.invoke([
        SystemMessage(content=ORCHESTRATOR_SYSTEM_PROMPT),
        HumanMessage(content=user_prompt)
    ]).content
    output = None

    for attempt in range(3):
        try:
            clean = llm_response.strip()
            if clean.startswith("```"):
                clean = clean.split("```")[1]
                if clean.startswith("json"):
                    clean = clean[4:]
            clean  = clean.strip()
            clean  = clean.replace('"null"', 'null')
            data   = json.loads(clean)
            output = OrchestratorOutput(**data)

            # Override lending decision programmatically
            output.lending_decision = determine_lending_decision(
                str(output.final_rating).replace("InternalRating.", "")
            )

            # Populate PD from Moody's table — never trust LLM for this
            final_rating_str = str(
                output.final_rating
            ).replace("InternalRating.", "")
            pd_12m, pd_3y        = get_pd_from_rating(final_rating_str)
            output.pd_12month    = pd_12m
            output.pd_3year      = pd_3y

            # Set human_input_received flag
            output.human_input_received = bool(human_input)
            break

        except Exception as e:
            last_error = str(e)
            if attempt < 2:
                llm_response = llm.invoke([
                    SystemMessage(content=ORCHESTRATOR_SYSTEM_PROMPT),
                    HumanMessage(content=(
                        f"Your output failed validation: {last_error}\n"
                        f"Fix and return ONLY valid JSON.\n"
                        f"Previous output: {llm_response}\n"
                        f"{ORCHESTRATOR_SCHEMA}"
                    ))
                ]).content

    if output is None:
        state["errors"].append(
            f"Orchestrator failed after 3 attempts: {last_error}"
        )
        state["current_stage"] = "orchestrator_failed"
        return state

    state["orchestrator_output"] = output.model_dump()
    state["current_stage"]       = "orchestrator_complete"

    print(f"Orchestrator complete — "
          f"final_rating: {output.final_rating}, "
          f"decision: {output.lending_decision}, "
          f"PD 12m: {output.pd_12month:.4f}, "
          f"PD 3y: {output.pd_3year:.4f}")

    return state


# --- Add macro data to state first ---
from src.data.macro import get_macro_context
macro_data = get_macro_context()
updated_state["macro_narrative"] = macro_data["narrative"]
updated_state["macro_snapshot"]  = macro_data["snapshot"]

# --- Test orchestrator ---
updated_state = run_orchestrator(updated_state)

print(f"\nStage: {updated_state['current_stage']}")
print(f"Errors: {updated_state['errors']}")
out = updated_state['orchestrator_output']
print(f"Final Rating: {out['final_rating']}")
print(f"Decision: {out['lending_decision']}")
print(f"PD 12m: {out['pd_12month']}")
print(f"PD 3y: {out['pd_3year']}")
print(f"\nNarrative preview:")
print(out['narrative_rationale'][:400])
print(f"\nTop Risk Factors:")
for rf in out['top_risk_factors']:
    print(f"  {rf['rank']}. [{rf['severity']}] {rf['factor']}")

Orchestrator complete — final_rating: InternalRating.AAA, decision: Approve, PD 12m: 0.0001, PD 3y: 0.0003

Stage: orchestrator_complete
Errors: []
Final Rating: InternalRating.AAA
Decision: Approve
PD 12m: 0.0001
PD 3y: 0.0003

Narrative preview:
The final rating of AAA is assigned based on Apple Inc.'s strong financial position, as evidenced by its low debt-to-EBITDA ratio, high interest coverage ratio, and high Altman Z-score. The company's ability to meet its financial obligations is demonstrated by its high interest coverage ratio and DSCR. The top risk factors, including regulatory changes and uncertain tax positions, are mitigated by

Top Risk Factors:
  1. [High] Regulatory changes
  2. [Medium] Uncertain tax positions
  3. [Medium] Global economic downturn
  4. [Low] Competition in services sector
  5. [Low] Default on senior unsecured obligations


c:\Users\Dev Vatsayan\credit-committee\venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='lending_decision', input_value='Approve', input_type=str])
  return self.__pydantic_serializer__.to_python(


In [13]:
# Fix determine_lending_decision to return enum
from src.schemas.models import LendingDecision

def determine_lending_decision(rating: str) -> LendingDecision:
    """Maps final rating to lending decision."""
    rating = rating.replace("InternalRating.", "")
    if rating in APPROVE_RATINGS:
        return LendingDecision.APPROVE
    elif rating in CONDITIONAL_RATINGS:
        return LendingDecision.CONDITIONAL
    return LendingDecision.DECLINE

# Quick test
print(determine_lending_decision("AAA"))   # Approve
print(determine_lending_decision("BB"))    # Conditional
print(determine_lending_decision("CCC"))   # Decline


LendingDecision.APPROVE
LendingDecision.CONDITIONAL
LendingDecision.DECLINE


In [14]:
# Test LangGraph wiring
# Build the full pipeline as a state graph

from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional, List

# Define state as TypedDict for LangGraph
# (LangGraph works better with TypedDict than Pydantic for state)
class PipelineState(TypedDict):
    # Input
    ticker          : str
    # Resolved identifiers
    legal_name      : str
    cik             : str
    sector          : str
    industry        : str
    market_cap      : Optional[float]
    # Financial data
    ratio_table     : Optional[dict]
    raw_financials  : Optional[dict]
    # Filing data
    mda             : Optional[str]
    risk_factors    : Optional[str]
    full_text       : Optional[str]
    # Transcript
    transcript_text : Optional[str]
    transcript_quarter : Optional[str]
    # Macro context
    macro_narrative : Optional[str]
    macro_snapshot  : Optional[dict]
    # Agent outputs
    research_analyst_output : Optional[dict]
    credit_analyst_output   : Optional[dict]
    devils_advocate_output  : Optional[dict]
    human_input             : Optional[str]
    orchestrator_output     : Optional[dict]
    # Pipeline status
    current_stage   : str
    errors          : List[str]
    warnings        : List[str]


def data_ingestion_node(state: dict) -> dict:
    """
    Runs all data modules and populates state before agents start.
    """
    from src.data.resolver import resolve_ticker
    from src.data.financials import get_financial_ratios
    from src.data.edgar import get_filing
    from src.data.transcripts import get_transcript
    from src.data.macro import get_macro_context

    ticker = state["ticker"]
    print(f"\nStarting data ingestion for {ticker}...")

    # Step 1: Resolve ticker
    resolved = resolve_ticker(ticker)
    if not resolved["status"]["success"]:
        state["errors"].append(f"Resolver failed: {resolved['status']['error']}")
        state["current_stage"] = "data_ingestion_failed"
        return state

    state["legal_name"] = resolved["legal_name"]
    state["cik"]        = resolved["cik"]
    state["sector"]     = resolved["sector"]
    state["industry"]   = resolved["industry"]
    state["market_cap"] = resolved["market_cap"]
    print(f"Resolved: {resolved['legal_name']} (CIK: {resolved['cik']})")

    # Step 2: Financial ratios
    ratio_data = get_financial_ratios(ticker)
    if ratio_data["status"]["success"]:
        state["ratio_table"]    = ratio_data
        state["raw_financials"] = ratio_data.get("raw", {})
        print(f"Financial data loaded")
    else:
        state["warnings"].append(
            f"Financial data failed: {ratio_data['status'].get('error')}"
        )

    # Step 3: EDGAR filing
    filing_data = get_filing(ticker, resolved["cik"])
    if filing_data["status"]["success"]:
        state["mda"]          = filing_data["mda"]
        state["risk_factors"] = filing_data["risk_factors"]
        state["full_text"]    = filing_data["full_text"]
        print(f"Filing loaded")
    else:
        state["warnings"].append(
            f"Filing failed: {filing_data['status'].get('error')}"
        )

    # Step 4: Transcript
    transcript_data = get_transcript(ticker)
    if transcript_data["status"]["success"]:
        state["transcript_text"]    = transcript_data["transcript_text"]
        state["transcript_quarter"] = transcript_data["quarter"]
        print(f"Transcript loaded: {transcript_data['quarter']}")
    else:
        state["warnings"].append(
            f"Transcript unavailable: {transcript_data['status'].get('error')}"
        )

    # Step 5: Macro context
    macro_data = get_macro_context()
    if macro_data["status"]["success"]:
        state["macro_narrative"] = macro_data["narrative"]
        state["macro_snapshot"]  = macro_data["snapshot"]
        print(f"Macro data loaded")
    else:
        state["warnings"].append("Macro data unavailable")

    state["current_stage"] = "data_ingestion_complete"
    print(f"Data ingestion complete")
    return state


def human_input_node(state: dict) -> dict:
    """Interrupt node — pauses for human input."""
    state["current_stage"] = "awaiting_human_input"
    return state


# Import agent functions
from src.agents.research_analyst import run_research_analyst
from src.agents.credit_analyst import run_credit_analyst
from src.agents.devil_advocate import run_devil_advocate
from src.agents.orchestrator import run_orchestrator


def build_graph():
    """Builds and compiles the LangGraph pipeline."""
    graph = StateGraph(dict)

    # Add nodes
    graph.add_node("data_ingestion",    data_ingestion_node)
    graph.add_node("research_analyst",  run_research_analyst)
    graph.add_node("credit_analyst",    run_credit_analyst)
    graph.add_node("devil_advocate",    run_devil_advocate)
    graph.add_node("human_input",       human_input_node)
    graph.add_node("orchestrator",      run_orchestrator)

    # Add edges
    graph.set_entry_point("data_ingestion")
    graph.add_edge("data_ingestion",   "research_analyst")
    graph.add_edge("research_analyst", "credit_analyst")
    graph.add_edge("credit_analyst",   "devil_advocate")
    graph.add_edge("devil_advocate",   "human_input")
    graph.add_edge("human_input",      "orchestrator")
    graph.add_edge("orchestrator",     END)

    # Compile with interrupt before orchestrator
    return graph.compile(interrupt_before=["orchestrator"])


# Test graph builds correctly
pipeline = build_graph()
print("Graph built successfully")
print("Nodes:", list(pipeline.get_graph().nodes.keys()))

Graph built successfully
Nodes: ['__start__', 'data_ingestion', 'research_analyst', 'credit_analyst', 'devil_advocate', 'human_input', 'orchestrator', '__end__']


In [15]:
# Test full pipeline end to end
# This runs all nodes except orchestrator (which waits for human input)

initial_state = {
    "ticker":                   "BA",  # Boeing — more interesting than AAPL
    "legal_name":               "",
    "cik":                      "",
    "sector":                   "",
    "industry":                 "",
    "market_cap":               None,
    "ratio_table":              None,
    "raw_financials":           None,
    "mda":                      None,
    "risk_factors":             None,
    "full_text":                None,
    "transcript_text":          None,
    "transcript_quarter":       None,
    "macro_narrative":          None,
    "macro_snapshot":           None,
    "research_analyst_output":  None,
    "credit_analyst_output":    None,
    "devils_advocate_output":   None,
    "human_input":              None,
    "orchestrator_output":      None,
    "current_stage":            "initialized",
    "errors":                   [],
    "warnings":                 []
}

# Run pipeline — will pause at orchestrator
print("Running pipeline...")
thread_config = {"configurable": {"thread_id": "test_run_1"}}

# Stream events so we can see progress
for event in pipeline.stream(initial_state, config=thread_config):
    for node_name, node_output in event.items():
        if node_name != "__end__":
            stage = node_output.get("current_stage", "unknown")
            errors = node_output.get("errors", [])
            print(f"Node '{node_name}' complete — stage: {stage}")
            if errors:
                print(f"  Errors: {errors}")

print("\nPipeline paused — waiting for human input")

Running pipeline...

Starting data ingestion for BA...
Resolved: The Boeing Company (CIK: 0000012927)
Financial data loaded
Filing loaded
Transcript loaded: None
Macro data loaded
Data ingestion complete
Node 'data_ingestion' complete — stage: data_ingestion_complete
Building new index for BA...
Research Analyst complete — tone: ManagementTone.CONFIDENT, going concern: False
Node 'research_analyst' complete — stage: research_analyst_complete
Credit Analyst complete — rating: InternalRating.AAA, primary concern: Current Ratio
Node 'credit_analyst' complete — stage: credit_analyst_complete
Loading existing index for BA...
Devil's Advocate complete — counter_rating: InternalRating.AA, default trigger: A combination of increased debt servicing costs and losses from operations in th
Node 'devil_advocate' complete — stage: devils_advocate_complete
Node 'human_input' complete — stage: awaiting_human_input


AttributeError: 'tuple' object has no attribute 'get'

In [16]:
# Fix stream reading
print("Running pipeline...")
thread_config = {"configurable": {"thread_id": "test_run_2"}}

final_state = None

for event in pipeline.stream(initial_state, config=thread_config):
    # LangGraph stream returns different formats
    # Handle both dict and tuple cases
    if isinstance(event, dict):
        for node_name, node_output in event.items():
            if node_name != "__end__" and isinstance(node_output, dict):
                stage  = node_output.get("current_stage", "unknown")
                errors = node_output.get("errors", [])
                print(f"Node '{node_name}' complete — stage: {stage}")
                if errors:
                    print(f"  Errors: {errors}")
                final_state = node_output

print(f"\nPipeline paused at: {final_state['current_stage'] if final_state else 'unknown'}")
print(f"Warnings: {final_state['warnings'] if final_state else []}")
print(f"Credit Analyst Rating: {final_state['credit_analyst_output']['preliminary_rating'] if final_state else 'N/A'}")
print(f"Devil Counter Rating: {final_state['devils_advocate_output']['counter_rating'] if final_state else 'N/A'}")

Running pipeline...

Starting data ingestion for BA...
Resolved: The Boeing Company (CIK: 0000012927)
Financial data loaded
Filing loaded
Transcript loaded: None
Macro data loaded
Data ingestion complete
Node 'data_ingestion' complete — stage: data_ingestion_complete
Loading existing index for BA...
Research Analyst complete — tone: ManagementTone.CONFIDENT, going concern: False
Node 'research_analyst' complete — stage: research_analyst_complete
Credit Analyst complete — rating: InternalRating.CCC, primary concern: Debt/EBITDA
Node 'credit_analyst' complete — stage: credit_analyst_complete
Loading existing index for BA...
Devil's Advocate complete — counter_rating: InternalRating.CC, default trigger: High debt levels and weak interest coverage leading to a liquidity crisis
Node 'devil_advocate' complete — stage: devils_advocate_complete
Node 'human_input' complete — stage: awaiting_human_input

Pipeline paused at: awaiting_human_input
Warnings: []
Credit Analyst Rating: InternalRating.

In [17]:
# Resume pipeline with human input
# Simulate analyst injecting a view before orchestrator runs

human_input_text = "Boeing has a strong defense backlog and government contracts that provide revenue stability. Consider this as a mitigant when assigning the final rating."

# Get current state from pipeline
current_state = pipeline.get_state(config=thread_config)

# Update human_input in state
pipeline.update_state(
    config=thread_config,
    values={"human_input": human_input_text}
)

print("Human input injected. Resuming pipeline...")

# Resume from interrupt
for event in pipeline.stream(None, config=thread_config):
    if isinstance(event, dict):
        for node_name, node_output in event.items():
            if node_name != "__end__" and isinstance(node_output, dict):
                stage  = node_output.get("current_stage", "unknown")
                errors = node_output.get("errors", [])
                print(f"Node '{node_name}' complete — stage: {stage}")
                if errors:
                    print(f"  Errors: {errors}")
                final_state = node_output

print(f"\nFinal stage: {final_state['current_stage']}")
out = final_state["orchestrator_output"]
print(f"Final Rating: {out['final_rating']}")
print(f"Decision: {out['lending_decision']}")
print(f"PD 12m: {out['pd_12month']}")
print(f"PD 3y: {out['pd_3year']}")
print(f"Human Input Received: {out['human_input_received']}")
print(f"\nTop Risk Factors:")
for rf in out["top_risk_factors"]:
    print(f"  {rf['rank']}. [{rf['severity']}] {rf['factor']}")
print(f"\nNarrative preview:")
print(out["narrative_rationale"][:400])

ValueError: No checkpointer set

In [18]:
# Rebuild graph with memory checkpointer
from langgraph.checkpoint.memory import MemorySaver

def build_graph_with_memory():
    """Builds and compiles the LangGraph pipeline with memory checkpointer."""
    graph = StateGraph(dict)

    graph.add_node("data_ingestion",   data_ingestion_node)
    graph.add_node("research_analyst", run_research_analyst)
    graph.add_node("credit_analyst",   run_credit_analyst)
    graph.add_node("devil_advocate",   run_devil_advocate)
    graph.add_node("human_input",      human_input_node)
    graph.add_node("orchestrator",     run_orchestrator)

    graph.set_entry_point("data_ingestion")
    graph.add_edge("data_ingestion",   "research_analyst")
    graph.add_edge("research_analyst", "credit_analyst")
    graph.add_edge("credit_analyst",   "devil_advocate")
    graph.add_edge("devil_advocate",   "human_input")
    graph.add_edge("human_input",      "orchestrator")
    graph.add_edge("orchestrator",     END)

    memory = MemorySaver()
    return graph.compile(
        checkpointer=memory,
        interrupt_before=["orchestrator"]
    )


# Rebuild and rerun pipeline from scratch with BA
pipeline = build_graph_with_memory()

initial_state = {
    "ticker":                   "BA",
    "legal_name":               "",
    "cik":                      "",
    "sector":                   "",
    "industry":                 "",
    "market_cap":               None,
    "ratio_table":              None,
    "raw_financials":           None,
    "mda":                      None,
    "risk_factors":             None,
    "full_text":                None,
    "transcript_text":          None,
    "transcript_quarter":       None,
    "macro_narrative":          None,
    "macro_snapshot":           None,
    "research_analyst_output":  None,
    "credit_analyst_output":    None,
    "devils_advocate_output":   None,
    "human_input":              None,
    "orchestrator_output":      None,
    "current_stage":            "initialized",
    "errors":                   [],
    "warnings":                 []
}

thread_config = {"configurable": {"thread_id": "ba_run_1"}}
final_state   = None

print("Running pipeline with memory checkpointer...")
for event in pipeline.stream(initial_state, config=thread_config):
    if isinstance(event, dict):
        for node_name, node_output in event.items():
            if node_name != "__end__" and isinstance(node_output, dict):
                stage  = node_output.get("current_stage", "unknown")
                errors = node_output.get("errors", [])
                print(f"Node '{node_name}' complete — stage: {stage}")
                if errors:
                    print(f"  Errors: {errors}")
                final_state = node_output

print(f"\nPipeline paused at: {final_state['current_stage'] if final_state else 'unknown'}")

Running pipeline with memory checkpointer...

Starting data ingestion for BA...
Resolved: The Boeing Company (CIK: 0000012927)
Financial data loaded
Filing loaded
Transcript loaded: None
Macro data loaded
Data ingestion complete
Node 'data_ingestion' complete — stage: data_ingestion_complete
Loading existing index for BA...
Research Analyst complete — tone: ManagementTone.CONFIDENT, going concern: False
Node 'research_analyst' complete — stage: research_analyst_complete
Credit Analyst complete — rating: InternalRating.CCC, primary concern: Debt/EBITDA
Node 'credit_analyst' complete — stage: credit_analyst_complete
Loading existing index for BA...
Devil's Advocate complete — counter_rating: InternalRating.CC, default trigger: High debt levels and increased interest payments leading to liquidity crisis
Node 'devil_advocate' complete — stage: devils_advocate_complete
Node 'human_input' complete — stage: awaiting_human_input

Pipeline paused at: awaiting_human_input


In [19]:
# Resume with human input
human_input_text = "Boeing has a strong defense backlog and government contracts that provide revenue stability. Consider this as a mitigant when assigning the final rating."

# Update state with human input
pipeline.update_state(
    config=thread_config,
    values={"human_input": human_input_text}
)

print("Human input injected. Resuming pipeline...")

final_state = None

for event in pipeline.stream(None, config=thread_config):
    if isinstance(event, dict):
        for node_name, node_output in event.items():
            if node_name != "__end__" and isinstance(node_output, dict):
                stage  = node_output.get("current_stage", "unknown")
                errors = node_output.get("errors", [])
                print(f"Node '{node_name}' complete — stage: {stage}")
                if errors:
                    print(f"  Errors: {errors}")
                final_state = node_output

print(f"\nFinal stage: {final_state['current_stage']}")
out = final_state["orchestrator_output"]
print(f"Final Rating: {out['final_rating']}")
print(f"Decision: {out['lending_decision']}")
print(f"PD 12m: {out['pd_12month']}")
print(f"PD 3y: {out['pd_3year']}")
print(f"Human Input Received: {out['human_input_received']}")
print(f"\nTop Risk Factors:")
for rf in out["top_risk_factors"]:
    print(f"  {rf['rank']}. [{rf['severity']}] {rf['factor']}")
print(f"\nNarrative preview:")
print(out["narrative_rationale"][:500])

Deserializing unregistered type src.schemas.models.ManagementTone from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('src.schemas.models', 'ManagementTone')]
Deserializing unregistered type src.schemas.models.InternalRating from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('src.schemas.models', 'InternalRating')]
Deserializing unregistered type src.schemas.models.RiskProbability from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('src.schemas.models', 'RiskProbability')]


Human input injected. Resuming pipeline...


KeyError: 'legal_name'

In [20]:
# Fix — get full state from checkpointer before resuming
current_state = pipeline.get_state(config=thread_config)
print("Current state keys:", list(current_state.values.keys()))
print("Current stage:", current_state.values.get("current_stage"))

Current state keys: ['human_input']
Current stage: None


In [21]:
# Correct approach — get full state snapshot first
snapshot = pipeline.get_state(config=thread_config)
print("Values:", list(snapshot.values.keys()))
print("Next nodes:", snapshot.next)

Values: ['human_input']
Next nodes: ('orchestrator',)


In [22]:
# Full clean run with memory checkpointer
# Use a fresh thread_id

thread_config = {"configurable": {"thread_id": "ba_run_clean"}}
final_state   = None

print("Running full pipeline with memory...")
for event in pipeline.stream(initial_state, config=thread_config):
    if isinstance(event, dict):
        for node_name, node_output in event.items():
            if node_name != "__end__" and isinstance(node_output, dict):
                stage = node_output.get("current_stage", "unknown")
                print(f"Node '{node_name}' — stage: {stage}")
                final_state = node_output

# Check full state is saved
snapshot = pipeline.get_state(config=thread_config)
print(f"\nCheckpointed keys: {list(snapshot.values.keys())}")
print(f"Next nodes: {snapshot.next}")
print(f"Legal name in state: {snapshot.values.get('legal_name', 'MISSING')}")

Running full pipeline with memory...

Starting data ingestion for BA...
Resolved: The Boeing Company (CIK: 0000012927)
Financial data loaded
Filing loaded
Transcript loaded: None
Macro data loaded
Data ingestion complete
Node 'data_ingestion' — stage: data_ingestion_complete
Loading existing index for BA...
Research Analyst complete — tone: ManagementTone.CONFIDENT, going concern: False
Node 'research_analyst' — stage: research_analyst_complete
Credit Analyst complete — rating: InternalRating.AAA, primary concern: Current Ratio
Node 'credit_analyst' — stage: credit_analyst_complete
Loading existing index for BA...
Devil's Advocate complete — counter_rating: InternalRating.AA, default trigger: A combination of increased debt servicing costs and losses from operations in th
Node 'devil_advocate' — stage: devils_advocate_complete
Node 'human_input' — stage: awaiting_human_input

Checkpointed keys: ['ticker', 'legal_name', 'cik', 'sector', 'industry', 'market_cap', 'ratio_table', 'raw_fina

In [23]:
# Resume with human input
human_input_text = "Boeing has a strong defense backlog and government contracts that provide revenue stability. Consider this as a mitigant when assigning the final rating."

pipeline.update_state(
    config=thread_config,
    values={"human_input": human_input_text}
)

print("Human input injected. Resuming pipeline...")

final_state = None

for event in pipeline.stream(None, config=thread_config):
    if isinstance(event, dict):
        for node_name, node_output in event.items():
            if node_name != "__end__" and isinstance(node_output, dict):
                stage  = node_output.get("current_stage", "unknown")
                errors = node_output.get("errors", [])
                print(f"Node '{node_name}' complete — stage: {stage}")
                if errors:
                    print(f"  Errors: {errors}")
                final_state = node_output

if final_state and final_state.get("orchestrator_output"):
    out = final_state["orchestrator_output"]
    print(f"\nFinal Rating: {out['final_rating']}")
    print(f"Decision: {out['lending_decision']}")
    print(f"PD 12m: {out['pd_12month']}")
    print(f"PD 3y: {out['pd_3year']}")
    print(f"Human Input Received: {out['human_input_received']}")
    print(f"\nTop Risk Factors:")
    for rf in out["top_risk_factors"]:
        print(f"  {rf['rank']}. [{rf['severity']}] {rf['factor']}")
    print(f"\nNarrative preview:")
    print(out["narrative_rationale"][:500])
else:
    print("No orchestrator output found")
    print("Final state stage:", final_state.get("current_stage") if final_state else "None")

Human input injected. Resuming pipeline...


KeyError: 'legal_name'

In [24]:
# Fix enum serialization — add to_serializable helper
import json

def to_serializable(obj):
    """Converts Pydantic output to plain JSON-serializable dict."""
    if hasattr(obj, 'model_dump'):
        raw = obj.model_dump()
    else:
        raw = obj
    return json.loads(json.dumps(raw, default=str))

# Test it works
from src.schemas.models import InternalRating, ManagementTone
test = {
    "rating": InternalRating.AAA,
    "tone": ManagementTone.CONFIDENT,
    "value": 0.68
}
result = to_serializable(test)
print(result)
print("Types:", {k: type(v).__name__ for k, v in result.items()})

{'rating': 'AAA', 'tone': 'Confident', 'value': 0.68}
Types: {'rating': 'str', 'tone': 'str', 'value': 'float'}


In [25]:
# Reimport all agents with the fix applied
import importlib
import src.agents.research_analyst as ra
import src.agents.credit_analyst as ca
import src.agents.devil_advocate as da
import src.agents.orchestrator as o

importlib.reload(ra)
importlib.reload(ca)
importlib.reload(da)
importlib.reload(o)

from src.agents.research_analyst import run_research_analyst
from src.agents.credit_analyst import run_credit_analyst
from src.agents.devil_advocate import run_devil_advocate
from src.agents.orchestrator import run_orchestrator

# Rebuild graph with memory
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

def build_graph_with_memory():
    graph = StateGraph(dict)
    graph.add_node("data_ingestion",   data_ingestion_node)
    graph.add_node("research_analyst", run_research_analyst)
    graph.add_node("credit_analyst",   run_credit_analyst)
    graph.add_node("devil_advocate",   run_devil_advocate)
    graph.add_node("human_input",      human_input_node)
    graph.add_node("orchestrator",     run_orchestrator)
    graph.set_entry_point("data_ingestion")
    graph.add_edge("data_ingestion",   "research_analyst")
    graph.add_edge("research_analyst", "credit_analyst")
    graph.add_edge("credit_analyst",   "devil_advocate")
    graph.add_edge("devil_advocate",   "human_input")
    graph.add_edge("human_input",      "orchestrator")
    graph.add_edge("orchestrator",     END)
    memory = MemorySaver()
    return graph.compile(
        checkpointer=memory,
        interrupt_before=["orchestrator"]
    )

pipeline     = build_graph_with_memory()
thread_config = {"configurable": {"thread_id": "ba_fixed_run"}}
final_state  = None

initial_state = {
    "ticker":                   "BA",
    "legal_name":               "",
    "cik":                      "",
    "sector":                   "",
    "industry":                 "",
    "market_cap":               None,
    "ratio_table":              None,
    "raw_financials":           None,
    "mda":                      None,
    "risk_factors":             None,
    "full_text":                None,
    "transcript_text":          None,
    "transcript_quarter":       None,
    "macro_narrative":          None,
    "macro_snapshot":           None,
    "research_analyst_output":  None,
    "credit_analyst_output":    None,
    "devils_advocate_output":   None,
    "human_input":              None,
    "orchestrator_output":      None,
    "current_stage":            "initialized",
    "errors":                   [],
    "warnings":                 []
}

print("Running pipeline...")
for event in pipeline.stream(initial_state, config=thread_config):
    if isinstance(event, dict):
        for node_name, node_output in event.items():
            if node_name != "__end__" and isinstance(node_output, dict):
                stage = node_output.get("current_stage", "unknown")
                print(f"Node '{node_name}' — stage: {stage}")
                final_state = node_output

snapshot = pipeline.get_state(config=thread_config)
print(f"\nPaused at: {snapshot.next}")
print(f"Legal name in checkpoint: {snapshot.values.get('legal_name', 'MISSING')}")

Running pipeline...

Starting data ingestion for BA...
Resolved: The Boeing Company (CIK: 0000012927)
Financial data loaded
Filing loaded
Transcript loaded: None
Macro data loaded
Data ingestion complete
Node 'data_ingestion' — stage: data_ingestion_complete
Loading existing index for BA...
Research Analyst complete — tone: ManagementTone.CONFIDENT, going concern: False
Node 'research_analyst' — stage: research_analyst_complete
Credit Analyst complete — rating: InternalRating.CCC, primary concern: Debt/EBITDA
Node 'credit_analyst' — stage: credit_analyst_complete
Loading existing index for BA...
Devil's Advocate complete — counter_rating: InternalRating.CC, default trigger: Inability to service debt due to high interest expenses and weak interest covera
Node 'devil_advocate' — stage: devils_advocate_complete
Node 'human_input' — stage: awaiting_human_input

Paused at: ('orchestrator',)
Legal name in checkpoint: The Boeing Company


In [26]:
# Resume with human input
human_input_text = "Boeing has a strong defense backlog and government contracts that provide revenue stability. Consider this as a mitigant when assigning the final rating."

pipeline.update_state(
    config=thread_config,
    values={"human_input": human_input_text}
)

print("Human input injected. Resuming...")

final_state = None

for event in pipeline.stream(None, config=thread_config):
    if isinstance(event, dict):
        for node_name, node_output in event.items():
            if node_name != "__end__" and isinstance(node_output, dict):
                stage  = node_output.get("current_stage", "unknown")
                errors = node_output.get("errors", [])
                print(f"Node '{node_name}' — stage: {stage}")
                if errors:
                    print(f"  Errors: {errors}")
                final_state = node_output

if final_state and final_state.get("orchestrator_output"):
    out = final_state["orchestrator_output"]
    print(f"\nFinal Rating: {out['final_rating']}")
    print(f"Decision: {out['lending_decision']}")
    print(f"PD 12m: {out['pd_12month']}")
    print(f"PD 3y: {out['pd_3year']}")
    print(f"Human Input Received: {out['human_input_received']}")
    print(f"\nTop Risk Factors:")
    for rf in out["top_risk_factors"]:
        print(f"  {rf['rank']}. [{rf['severity']}] {rf['factor']}")
    print(f"\nNarrative preview:")
    print(out["narrative_rationale"][:500])
else:
    print("No orchestrator output")
    if final_state:
        print("Errors:", final_state.get("errors", []))

Human input injected. Resuming...


KeyError: 'legal_name'

In [27]:
# Reload orchestrator and rebuild graph
importlib.reload(o)
from src.agents.orchestrator import run_orchestrator

pipeline      = build_graph_with_memory()
thread_config = {"configurable": {"thread_id": "ba_fixed_run_2"}}
final_state   = None

print("Running pipeline...")
for event in pipeline.stream(initial_state, config=thread_config):
    if isinstance(event, dict):
        for node_name, node_output in event.items():
            if node_name != "__end__" and isinstance(node_output, dict):
                stage = node_output.get("current_stage", "unknown")
                print(f"Node '{node_name}' — stage: {stage}")
                final_state = node_output

print(f"\nPaused at: {pipeline.get_state(config=thread_config).next}")

# Inject human input and resume
pipeline.update_state(
    config=thread_config,
    values={"human_input": "Boeing has a strong defense backlog. Consider this as a mitigant."}
)

print("Resuming with human input...")
for event in pipeline.stream(None, config=thread_config):
    if isinstance(event, dict):
        for node_name, node_output in event.items():
            if node_name != "__end__" and isinstance(node_output, dict):
                stage = node_output.get("current_stage", "unknown")
                print(f"Node '{node_name}' — stage: {stage}")
                final_state = node_output

if final_state and final_state.get("orchestrator_output"):
    out = final_state["orchestrator_output"]
    print(f"\nFinal Rating: {out['final_rating']}")
    print(f"Decision: {out['lending_decision']}")
    print(f"PD 12m: {out['pd_12month']}")
    print(f"PD 3y: {out['pd_3year']}")
    print(f"Human Input Received: {out['human_input_received']}")
    print(f"\nTop Risk Factors:")
    for rf in out["top_risk_factors"]:
        print(f"  {rf['rank']}. [{rf['severity']}] {rf['factor']}")
    print(f"\nNarrative preview:")
    print(out["narrative_rationale"][:400])

Running pipeline...

Starting data ingestion for BA...
Resolved: The Boeing Company (CIK: 0000012927)
Financial data loaded
Filing loaded
Transcript loaded: None
Macro data loaded
Data ingestion complete
Node 'data_ingestion' — stage: data_ingestion_complete
Loading existing index for BA...
Research Analyst complete — tone: ManagementTone.CONFIDENT, going concern: False
Node 'research_analyst' — stage: research_analyst_complete
Credit Analyst complete — rating: InternalRating.AAA, primary concern: None
Node 'credit_analyst' — stage: credit_analyst_complete
Loading existing index for BA...
Devil's Advocate complete — counter_rating: InternalRating.AA, default trigger: A combination of rising interest rates and production disruptions in the commerc
Node 'devil_advocate' — stage: devils_advocate_complete
Node 'human_input' — stage: awaiting_human_input

Paused at: ('orchestrator',)
Resuming with human input...
Orchestrator complete — final_rating: InternalRating.BBB, decision: LendingDeci

In [28]:
# Quick test — reload agents and test Credit Analyst on Boeing directly
importlib.reload(ca)
from src.agents.credit_analyst import run_credit_analyst

# Simulate Boeing state
boeing_state = {
    "ticker":     "BA",
    "legal_name": "The Boeing Company",
    "sector":     "Industrials",
    "ratio_table": {
        "latest_year": "2025-12-31",
        "ratios": {
            "debt_to_ebitda":     7.4,
            "interest_coverage":  1.95,
            "current_ratio":      1.19,
            "fcf_to_debt":       -0.03,
            "net_debt_to_ebitda": 5.83,
            "dscr":               0.02,
            "altman_z":           2.89,
            "altman_zone":        "GREY"
        }
    },
    "research_analyst_output": {
        "business_summary": "Boeing is a major aerospace and defense company.",
        "liquidity_assessment": "Tight liquidity with high debt levels.",
        "management_tone": "Cautious",
        "going_concern_flag": False
    },
    "errors":   [],
    "warnings": []
}

result = run_credit_analyst(boeing_state)
print("Rating:", result["credit_analyst_output"]["preliminary_rating"])
print("Primary concern:", result["credit_analyst_output"]["primary_concern"])
print("Rationale:", result["credit_analyst_output"]["preliminary_rating_rationale"][:300])

Credit Analyst complete — rating: InternalRating.CCC, primary concern: Debt/EBITDA
Rating: CCC
Primary concern: Debt/EBITDA
Rationale: The preliminary rating of CCC is assigned due to the company's high debt levels, weak interest coverage, and low Altman Z-score. Although the current ratio is slightly above 1x, the overall credit quality is compromised by the significant debt burden and deteriorating trends in leverage, coverage, a


In [29]:
# Test AAPL and NVDA ratings
test_cases = [
    {
        "ticker": "AAPL",
        "legal_name": "Apple Inc.",
        "sector": "Technology",
        "ratio_table": {
            "latest_year": "2025-09-30",
            "ratios": {
                "debt_to_ebitda":     0.68,
                "interest_coverage":  33.83,
                "current_ratio":      0.89,
                "fcf_to_debt":        1.0,
                "net_debt_to_ebitda": 0.43,
                "dscr":               1.13,
                "altman_z":           29.77,
                "altman_zone":        "SAFE"
            }
        },
        "research_analyst_output": {
            "business_summary": "Apple is a technology company.",
            "liquidity_assessment": "Strong liquidity position.",
            "management_tone": "Confident",
            "going_concern_flag": False
        },
        "errors": [], "warnings": []
    },
    {
        "ticker": "NVDA",
        "legal_name": "NVIDIA Corporation",
        "sector": "Technology",
        "ratio_table": {
            "latest_year": "2025-01-31",
            "ratios": {
                "debt_to_ebitda":     0.5,
                "interest_coverage":  50.0,
                "current_ratio":      4.0,
                "fcf_to_debt":        2.0,
                "net_debt_to_ebitda": 0.2,
                "dscr":               3.0,
                "altman_z":           15.0,
                "altman_zone":        "SAFE"
            }
        },
        "research_analyst_output": {
            "business_summary": "NVIDIA is a semiconductor company.",
            "liquidity_assessment": "Extremely strong liquidity.",
            "management_tone": "Confident",
            "going_concern_flag": False
        },
        "errors": [], "warnings": []
    }
]

for test in test_cases:
    result = run_credit_analyst(test)
    print(f"\n{test['ticker']}:")
    print(f"  Rating: {result['credit_analyst_output']['preliminary_rating']}")
    print(f"  Primary concern: {result['credit_analyst_output']['primary_concern']}")
    

Credit Analyst complete — rating: InternalRating.AA, primary concern: Current Ratio

AAPL:
  Rating: AA
  Primary concern: Current Ratio
Credit Analyst complete — rating: InternalRating.AAA, primary concern: None

NVDA:
  Rating: AAA
  Primary concern: None
